
# PCB - Ingestion Data Pipeline 

This is the pipeline we will be building. We ingest 2 datasets, namely:

* The raw images (jpg) containing PCB
* The label, the type of anomalies saved as CSV files

We will first focus on building a data pipeline to incrementally load this data and create a final Gold table.

This table will then be used to train a ML Classification model to learn to detect anomalies in our images in real time!

*Note that this demo leverages the standard spark API. You could also implement this same pipeline in pure SQL leveraging Lakeflow Pipelines. For more details on Lakeflow Pipelines, install `dbdemos.install('pipeline-bike')`*

<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=data-science&org_id=7474654018043568&notebook=%2F01-ingestion-and-ETL&demo_name=computer-vision-pcb&event=VIEW&path=%2F_dbdemos%2Fdata-science%2Fcomputer-vision-pcb%2F01-ingestion-and-ETL&version=1&user_hash=bbdbed5f785ffc3e7d74bf8bb0e97e3e6421d6ad205d21255f112e10dfba5ca0">

In [0]:
%pip install databricks-sdk mlflow==3.14.0
dbutils.library.restartPython()

In [0]:
%run ./_resources/00-init $reset_all_data=false

In [0]:
print(f"Training data has been installed in the volume {volume_folder}")

## Reviewing the incoming dataset

The dataset was downloaded for you automatically and is available in your cloud storage folder. Let's explore the data:

In [0]:
display(dbutils.fs.ls(f"{volume_folder}/images/Normal/"))
display(dbutils.fs.ls(f"{volume_folder}/labels/"))
display(dbutils.fs.head(f"{volume_folder}/labels/image_anno.csv"))

### PCB images inspection

We can display images with `matplotlib` in a native python way.

Let us investigate what a normal image looks like, and then one with an anomaly.

In [0]:
volume_folder

In [0]:
from PIL import Image
import matplotlib.pyplot as plt

def display_image(path, dpi=300):
    img = Image.open(path)
    width, height = img.size
    plt.figure(figsize=(width / dpi, height / dpi))
    plt.imshow(img, interpolation="nearest", aspect="auto")


display_image(f"{volume_folder}/images/Normal/0000.JPG")
display_image(f"{volume_folder}/images/Anomaly/000.JPG")

## Ingesting raw images with Databricks Autoloader

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/computer-vision/deeplearning-cv-pcb-flow-1.png?raw=true" width="700px" style="float: right"/>

The first step is to load the individual JPG images. This can be quite challenging at scale, especially for incremental load (consume only the new ones).

Databricks Autoloader can easily handle all type of format and make it very easy to ingest new datasets.

Autoloader will guarantee that only new files are being processed while scaling with millions of individual images. 

### Load binary files with Auto Loader

We can now use the Auto Loader to load images, and a spark function to create the label column. Autoloader will automatically create the table and tune it accordingly, disabling compression for binary among other.

We can also very easily display the content of the images and the labels as a table.

In [0]:
(spark.readStream.format("cloudFiles")
                 .option("cloudFiles.format", "binaryFile")
                 .option("pathGlobFilter", "*.JPG")
                 .option("recursiveFileLookup", "true")
                 .option("cloudFiles.schemaLocation", f"{volume_folder}/stream/pcb_schema")
                 .option("cloudFiles.maxFilesPerTrigger", 200)
                 .load(f"{volume_folder}/images/")
    .withColumn("filename", F.substring_index(col("path"), "/", -1))
    .writeStream.trigger(availableNow=True)
                .option("checkpointLocation", f"{volume_folder}/stream/pcb_checkpoint")
                .toTable("pcb_images").awaitTermination())

spark.sql("ALTER TABLE pcb_images OWNER TO `account users`")
display(spark.table("pcb_images"))

### Load CSV label files with Auto Loader
CSV files can easily be loaded using Databricks [Auto Loader](https://docs.databricks.com/ingestion/auto-loader/index.html), including schema inference and evolution.

In [0]:
(spark.readStream.format("cloudFiles")
                 .option("cloudFiles.format", "csv")
                 .option("header", True)
                 .option("cloudFiles.schemaLocation", f"{volume_folder}/stream/labels_schema")
                 .load(f"{volume_folder}/labels/")
      .withColumn("filename", F.substring_index(col("image"), "/", -1))
      .select("filename", "label")
      .withColumnRenamed("label", "labelDetail")
      .writeStream.trigger(availableNow=True)
                  .option("checkpointLocation", f"{volume_folder}/stream/labels_checkpoint")
                  .toTable("pcb_labels").awaitTermination())

spark.sql("ALTER TABLE pcb_labels SET OWNER TO `account users`")
display(spark.table("pcb_labels"))

## Let's now merge the labels and the images tables

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/computer-vision/deeplearning-cv-pcb-flow-2.png?raw=true" width="700px" style="float: right"/>

Note that we're working with delta tables to make the ingestion simple. 

You don't have to worry about individual small images anymore.

We can do the join operation either in python or SQL.

In [0]:
%sql
CREATE OR REPLACE TABLE training_dataset AS 
  (SELECT 
      *, 
      CASE WHEN labelDetail = 'normal' THEN 'normal' ELSE 'damaged' END as label
   FROM 
      pcb_images 
    INNER JOIN pcb_labels USING (filename)
  );

ALTER TABLE training_dataset SET OWNER TO `account users`;

SELECT * FROM training_dataset LIMIT 10;

## Final step: Preparing and augmenting our image dataset for DL Fine Tuning

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/computer-vision/deeplearning-cv-pcb-flow-3.png?raw=true" width="700px" style="float: right"/>

Note that we're working with tables. We can do this transformation in python or SQL.

Some transformation on image can be expensive. We can leverage Spark to distribute some image pre-processing first.

In this example, we will do the following:
- crop the image in the center to make them square (the model we use for fine-tuning take square images)
- resize our image to smaller a resolution (256x256) as our models won't use images with high resolution. 

We will also augment our dataset to add more "damaged" items as we have here something fairly imbalanced (only 1 on 10 item has an anomaly). <br/>
It looks like our system takes pcb pictures upside/down without preference and that's how our inferences will be. Let's then flip all the damaged images horizontally and add them back in our dataset.

*Note: if you're using deltatorch, you can directly split your test/training dataset and add an id column for each here directly. For more detail open the 04-ADVANCED-pytorch-training-and-inference notebook*

In [0]:
from PIL import Image
import io
from pyspark.sql.functions import pandas_udf
IMAGE_RESIZE = 256

#Resize UDF function
@pandas_udf("binary")
def resize_image_udf(content_series):
  def resize_image(content):
    """resize image and serialize back as jpeg"""
    #Load the PIL image
    image = Image.open(io.BytesIO(content))
    width, height = image.size   # Get dimensions
    new_size = min(width, height)
    # Crop the center of the image
    image = image.crop(((width - new_size)/2, (height - new_size)/2, (width + new_size)/2, (height + new_size)/2))
    #Resize to the new resolution
    image = image.resize((IMAGE_RESIZE, IMAGE_RESIZE), Image.NEAREST)
    #Save back as jpeg
    output = io.BytesIO()
    image.save(output, format='JPEG')
    return output.getvalue()
  return content_series.apply(resize_image)


# add the metadata to enable the image preview
image_meta = {"spark.contentAnnotation" : '{"mimeType": "image/jpeg"}'}

(spark.table("training_dataset")
      .withColumn("sort", F.rand()).orderBy("sort").drop('sort') #shuffle the DF
      .withColumn("content", resize_image_udf(col("content")).alias("content", metadata=image_meta))
      .write.mode('overwrite').saveAsTable("training_dataset_augmented"))

spark.sql("ALTER TABLE training_dataset_augmented OWNER TO `account users`")

In [0]:
import PIL
@pandas_udf("binary")
def flip_image_horizontal_udf(content_series):
  def flip_image(content):
    """resize image and serialize back as jpeg"""
    #Load the PIL image
    image = Image.open(io.BytesIO(content))
    #Flip
    image = image.transpose(PIL.Image.FLIP_TOP_BOTTOM)
    #Save back as jpeg
    output = io.BytesIO()
    image.save(output, format='JPEG')
    return output.getvalue()
  return content_series.apply(flip_image)

(spark.table("training_dataset_augmented")
    .filter("label == 'damaged'")
    .withColumn("content", flip_image_horizontal_udf(col("content")).alias("content", metadata=image_meta))
    .write.mode('append').saveAsTable("training_dataset_augmented"))

In [0]:
%sql
SELECT
  label,
  count(*)
FROM
  training_dataset_augmented
GROUP BY
  label


### Our dataset is ready for our Data Science team

That's it! We have now deployed a production-ready ingestion pipeline.

Our images are incrementally ingested and joined with our label dataset.

Let's see how this data can be used by a Data Scientist to [build our Computer Vision model]($./02-huggingface-model-training).